# 🤖 SmartSpend — AI Engineer: Deep Learning Model
**Coding Camp 2026 | CC26-PSU146**

Notebook ini dibuat oleh AI Engineer untuk memenuhi checklist MVP:
- ✅ Deep Learning dengan **TensorFlow Functional API**
- ✅ Komponen kustom: **Custom Layer + Custom Callback**
- ✅ Menyimpan model dalam format **`.keras` (SavedModel)**
- ✅ Kode **inference** siap produksi

---
**Task:** Multi-output model:
- **Output 1 (Klasifikasi):** Prediksi `label_rekomendasi` (Keuangan Sehat / Cukup Baik / Perlu Perbaikan)
- **Output 2 (Regresi):** Prediksi `tabungan_bulanan` yang ideal

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import joblib
import json
import os

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')
tf.random.set_seed(42)
np.random.seed(42)

## 2. Load & Preprocessing Data
> Data sudah di-EDA oleh tim Data Scientist. AI Engineer melakukan preprocessing untuk kebutuhan Deep Learning.

In [ ]:
# Load dataset
df = pd.read_csv('dataset_finansial.csv')
print(f'Dataset shape: {df.shape}')
print(f"\nLabel distribution:")
print(df['label_rekomendasi'].value_counts())
df.head()

In [ ]:
# ============================================================
# PREPROCESSING
# ============================================================

# 1. Drop kolom ID (tidak relevan untuk modeling)
df = df.drop(columns=['id'])

# 2. Simpan provinsi sebagai fitur tambahan (label encode)
# Karena banyak kategori, kita gunakan frekuensi encoding
provinsi_freq = df['provinsi'].value_counts(normalize=True).to_dict()
df['provinsi_freq'] = df['provinsi'].map(provinsi_freq)
df = df.drop(columns=['provinsi'])

# 3. Label Encoding untuk kolom kategorikal
le_dict = {}
cat_cols = ['klasifikasi_wilayah', 'jenis_kelamin', 'pendidikan_terakhir',
            'status_pekerjaan', 'status_pernikahan']

for col in cat_cols:
    le = LabelEncoder()
    # Isi missing value dengan modus sebelum encode
    df[col] = df[col].fillna(df[col].mode()[0])
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# 4. Isi missing value numerik dengan median
num_cols = df.select_dtypes(include='number').columns.tolist()
# Exclude target kolom dari imputasi sekarang
target_cols = ['label_rekomendasi', 'tabungan_bulanan']
num_features = [c for c in num_cols if c not in target_cols and c != 'tabungan_bulanan']

for col in df.select_dtypes(include='number').columns:
    if col != 'label_rekomendasi':  # label_rekomendasi masih string
        df[col] = df[col].fillna(df[col].median())

# 5. Encode target label klasifikasi
le_label = LabelEncoder()
df['label_encoded'] = le_label.fit_transform(df['label_rekomendasi'])
print('Label classes:', le_label.classes_)
print('Label mapping:', dict(zip(le_label.classes_, le_label.transform(le_label.classes_))))

df.shape

In [ ]:
# ============================================================
# DEFINISI FITUR (INPUT) DAN TARGET (OUTPUT)
# ============================================================

# Fitur input untuk model
FEATURE_COLS = [
    # Demografi
    'usia', 'jenis_kelamin', 'klasifikasi_wilayah', 'pendidikan_terakhir',
    'status_pekerjaan', 'status_pernikahan', 'jumlah_tanggungan', 'provinsi_freq',
    # Keuangan utama
    'pendapatan_bulanan', 'total_pengeluaran', 'rasio_tabungan_persen',
    'skor_literasi_keuangan', 'cicilan_hutang',
    # Pengeluaran per kategori
    'pengeluaran_makanan_pokok', 'pengeluaran_lauk_pauk', 'pengeluaran_sayur_buah',
    'pengeluaran_jajan_makan_luar', 'pengeluaran_rokok_tembakau',
    'pengeluaran_perumahan_listrik', 'pengeluaran_pakaian', 'pengeluaran_kesehatan',
    'pengeluaran_pendidikan', 'pengeluaran_transportasi', 'pengeluaran_komunikasi',
    'pengeluaran_hiburan_rekreasi'
]

# Target 1: Klasifikasi kondisi keuangan
TARGET_CLASS  = 'label_encoded'
# Target 2: Regresi nominal tabungan ideal
TARGET_REGRES = 'tabungan_bulanan'

X = df[FEATURE_COLS].values.astype(np.float32)
y_class  = df[TARGET_CLASS].values.astype(np.int32)
y_regres = df[TARGET_REGRES].values.astype(np.float32)

print(f'X shape       : {X.shape}')
print(f'y_class shape : {y_class.shape}  | unique: {np.unique(y_class)}')
print(f'y_regres shape: {y_regres.shape} | range: {y_regres.min():.0f} - {y_regres.max():.0f}')

In [ ]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT  (70 / 15 / 15)
# ============================================================

X_train, X_temp, y_c_train, y_c_temp, y_r_train, y_r_temp = train_test_split(
    X, y_class, y_regres, test_size=0.30, random_state=42, stratify=y_class
)
X_val, X_test, y_c_val, y_c_test, y_r_val, y_r_test = train_test_split(
    X_temp, y_c_temp, y_r_temp, test_size=0.50, random_state=42, stratify=y_c_temp
)

# Normalisasi fitur (fit hanya pada training data)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

# Normalisasi target regresi (log-scale agar training lebih stabil)
y_r_train_log = np.log1p(y_r_train)
y_r_val_log   = np.log1p(y_r_val)
y_r_test_log  = np.log1p(y_r_test)

print(f'Train : {X_train.shape[0]} samples')
print(f'Val   : {X_val.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')

# Simpan scaler untuk inference
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le_dict, 'label_encoders.pkl')
joblib.dump(le_label, 'label_encoder_target.pkl')
joblib.dump(provinsi_freq, 'provinsi_freq.pkl')
print('\nScaler & encoders saved.')

## 3. Custom Components
### 3a. Custom Layer — `AttentionLayer`
Layer ini menambahkan **mekanisme attention** sederhana agar model bisa memfokuskan perhatian pada fitur keuangan yang paling penting untuk setiap user.

In [ ]:
class FinancialAttentionLayer(layers.Layer):
    """
    Custom Layer: Financial Attention
    ----------------------------------
    Memberikan bobot (attention weight) berbeda pada setiap fitur
    keuangan, sehingga model bisa fokus pada fitur yang paling
    relevan untuk setiap pengguna secara dinamis.

    Cara kerja:
    1. Hitung skor relevansi tiap fitur via dense layer kecil
    2. Normalisasi dengan softmax → attention weights
    3. Kalikan input dengan attention weights (element-wise)
    """

    def __init__(self, units=64, **kwargs):
        super(FinancialAttentionLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        # Bobot untuk menghitung attention score
        self.W_attention = self.add_weight(
            name='W_attention',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b_attention = self.add_weight(
            name='b_attention',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        # Proyeksi kembali ke dimensi input
        self.W_out = self.add_weight(
            name='W_out',
            shape=(self.units, input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        # Hitung attention score
        score = tf.nn.tanh(tf.matmul(inputs, self.W_attention) + self.b_attention)
        score = tf.matmul(score, self.W_out)
        # Normalisasi dengan softmax
        attention_weights = tf.nn.softmax(score, axis=-1)
        # Terapkan attention ke input
        attended = inputs * attention_weights
        return attended

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config


print('✅ Custom Layer: FinancialAttentionLayer berhasil didefinisikan')

### 3b. Custom Callback — `SmartSpendCallback`
Callback ini memantau training secara real-time dan otomatis menyimpan model terbaik.

In [ ]:
class SmartSpendCallback(Callback):
    """
    Custom Callback: SmartSpend Training Monitor
    ---------------------------------------------
    Memantau proses training dan:
    1. Mencetak ringkasan performa tiap epoch
    2. Menyimpan model terbaik otomatis berdasarkan val_loss
    3. Menghentikan training jika val_loss tidak membaik (early stopping)
    4. Menyimpan history training ke file JSON
    """

    def __init__(self, patience=10, save_path='best_model.keras'):
        super().__init__()
        self.patience   = patience
        self.save_path  = save_path
        self.best_loss  = np.inf
        self.wait       = 0
        self.history_log = []

    def on_train_begin(self, logs=None):
        print('=' * 60)
        print('🚀 SmartSpend Model Training Dimulai')
        print('=' * 60)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_loss       = logs.get('val_loss', np.inf)
        val_class_acc  = logs.get('val_classification_output_accuracy', 0)
        val_regres_mae = logs.get('val_regression_output_mae', 0)

        # Simpan ke history log
        self.history_log.append({
            'epoch'        : epoch + 1,
            'val_loss'     : round(float(val_loss), 4),
            'val_class_acc': round(float(val_class_acc), 4),
            'val_mae'      : round(float(val_regres_mae), 4)
        })

        # Print setiap 5 epoch
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:3d} | val_loss: {val_loss:.4f} | '
                  f'val_acc: {val_class_acc:.4f} | val_mae: {val_regres_mae:.0f}')

        # Simpan model terbaik
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.wait      = 0
            self.model.save(self.save_path)
            if (epoch + 1) % 5 == 0:
                print(f'  💾 Model terbaik disimpan (val_loss: {val_loss:.4f})')
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                print(f'\n⛔ Early stopping pada epoch {epoch+1} '
                      f'(tidak ada perbaikan selama {self.patience} epoch)')

    def on_train_end(self, logs=None):
        # Simpan history ke JSON
        with open('training_history.json', 'w') as f:
            json.dump(self.history_log, f, indent=2)
        print('\n' + '=' * 60)
        print(f'✅ Training selesai! Best val_loss: {self.best_loss:.4f}')
        print(f'📁 History disimpan di training_history.json')
        print('=' * 60)


print('✅ Custom Callback: SmartSpendCallback berhasil didefinisikan')

## 4. Membangun Model dengan TensorFlow Functional API

Model ini adalah **multi-output neural network** yang:
- Menggunakan **Functional API** (bukan Sequential)
- Memiliki **shared backbone** (layer bersama)
- Memiliki **dua output head**: klasifikasi + regresi
- Menggunakan **Custom Layer** `FinancialAttentionLayer`

In [ ]:
def build_smartspend_model(input_dim, num_classes=3):
    """
    SmartSpend Deep Learning Model — TensorFlow Functional API
    -----------------------------------------------------------
    Arsitektur:
      Input → FinancialAttentionLayer → Shared Backbone
                                              |
                         ┌────────────────────┴────────────────────┐
                    Classification Head                    Regression Head
                  (Kondisi Keuangan)               (Nominal Tabungan Ideal)
    """

    # ── INPUT LAYER ──────────────────────────────────────────
    inputs = keras.Input(shape=(input_dim,), name='financial_features')

    # ── CUSTOM ATTENTION LAYER ────────────────────────────────
    x = FinancialAttentionLayer(units=64, name='financial_attention')(inputs)

    # ── SHARED BACKBONE ──────────────────────────────────────
    # Layer 1
    x = layers.Dense(256, name='backbone_dense_1')(x)
    x = layers.BatchNormalization(name='backbone_bn_1')(x)
    x = layers.Activation('relu', name='backbone_relu_1')(x)
    x = layers.Dropout(0.3, name='backbone_drop_1')(x)

    # Layer 2
    x = layers.Dense(128, name='backbone_dense_2')(x)
    x = layers.BatchNormalization(name='backbone_bn_2')(x)
    x = layers.Activation('relu', name='backbone_relu_2')(x)
    x = layers.Dropout(0.3, name='backbone_drop_2')(x)

    # Layer 3 (Residual connection sederhana)
    x_skip = layers.Dense(64, name='skip_projection')(x)
    x = layers.Dense(64, name='backbone_dense_3')(x)
    x = layers.BatchNormalization(name='backbone_bn_3')(x)
    x = layers.Activation('relu', name='backbone_relu_3')(x)
    x = layers.Add(name='residual_add')([x, x_skip])  # Residual

    shared_output = x  # Output dari shared backbone

    # ── CLASSIFICATION HEAD ───────────────────────────────────
    # Memprediksi label kondisi keuangan
    clf = layers.Dense(32, activation='relu', name='clf_dense_1')(shared_output)
    clf = layers.Dropout(0.2, name='clf_drop')(clf)
    clf_output = layers.Dense(
        num_classes, activation='softmax', name='classification_output'
    )(clf)

    # ── REGRESSION HEAD ──────────────────────────────────────
    # Memprediksi nominal tabungan ideal (dalam log-scale)
    reg = layers.Dense(32, activation='relu', name='reg_dense_1')(shared_output)
    reg = layers.Dropout(0.2, name='reg_drop')(reg)
    reg_output = layers.Dense(
        1, activation='linear', name='regression_output'
    )(reg)

    # ── BUILD MODEL ───────────────────────────────────────────
    model = Model(
        inputs=inputs,
        outputs=[clf_output, reg_output],
        name='SmartSpend_DL_Model'
    )
    return model


# Build model
model = build_smartspend_model(input_dim=X_train.shape[1], num_classes=3)
model.summary()

## 5. Compile Model

In [ ]:
# Hitung class weight untuk mengatasi imbalance dataset
# (Perlu Perbaikan hanya 169 dari 2150 data)
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_c_train),
    y=y_c_train
)
class_weight_dict = dict(enumerate(class_weights_arr))
print('Class weights:', class_weight_dict)
print('Classes:', le_label.classes_)

# Compile model dengan multi-loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'classification_output': 'sparse_categorical_crossentropy',
        'regression_output'    : 'mse'
    },
    loss_weights={
        'classification_output': 1.0,   # Bobot loss klasifikasi
        'regression_output'    : 0.1    # Bobot loss regresi (lebih kecil karena skala berbeda)
    },
    metrics={
        'classification_output': ['accuracy'],
        'regression_output'    : ['mae']
    }
)

print('\n✅ Model berhasil dikompilasi')

## 6. Training Model

In [ ]:
# Inisialisasi Custom Callback
smartspend_cb = SmartSpendCallback(
    patience=15,
    save_path='smartspend_best_model.keras'
)

# Learning rate scheduler
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=0
)

# Training
history = model.fit(
    X_train,
    {
        'classification_output': y_c_train,
        'regression_output'    : y_r_train_log
    },
    validation_data=(
        X_val,
        {
            'classification_output': y_c_val,
            'regression_output'    : y_r_val_log
        }
    ),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,  # Untuk mengatasi imbalance
    callbacks=[smartspend_cb, lr_scheduler],
    verbose=0  # Output dihandle oleh Custom Callback
)

## 7. Evaluasi Model

In [ ]:
# ── PLOT TRAINING HISTORY ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('SmartSpend — Training History', fontsize=14, fontweight='bold')

# Total Loss
axes[0].plot(history.history['loss'],     label='Train Loss', color='#3498db')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#e74c3c')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Classification Accuracy
axes[1].plot(history.history['classification_output_accuracy'],
             label='Train Acc', color='#2ecc71')
axes[1].plot(history.history['val_classification_output_accuracy'],
             label='Val Acc', color='#e67e22')
axes[1].set_title('Klasifikasi Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# Regression MAE
axes[2].plot(history.history['regression_output_mae'],
             label='Train MAE', color='#9b59b6')
axes[2].plot(history.history['val_regression_output_mae'],
             label='Val MAE', color='#1abc9c')
axes[2].set_title('Regresi MAE (log-scale)')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('training_history_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EVALUASI PADA TEST SET ────────────────────────────────────────
# Load model terbaik
best_model = keras.models.load_model(
    'smartspend_best_model.keras',
    custom_objects={'FinancialAttentionLayer': FinancialAttentionLayer}
)

# Prediksi
clf_pred_prob, reg_pred_log = best_model.predict(X_test, verbose=0)
clf_pred_label = np.argmax(clf_pred_prob, axis=1)
reg_pred_rupiah = np.expm1(reg_pred_log.flatten())  # Invers log1p

print('=' * 55)
print('📊 EVALUASI KLASIFIKASI KONDISI KEUANGAN')
print('=' * 55)
print(classification_report(
    y_c_test,
    clf_pred_label,
    target_names=le_label.classes_
))

# Confusion Matrix
cm = confusion_matrix(y_c_test, clf_pred_label)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_label.classes_,
            yticklabels=le_label.classes_)
plt.title('Confusion Matrix — Kondisi Keuangan', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 55)
print('📊 EVALUASI REGRESI NOMINAL TABUNGAN')
print('=' * 55)
mae  = np.mean(np.abs(reg_pred_rupiah - y_r_test))
rmse = np.sqrt(np.mean((reg_pred_rupiah - y_r_test) ** 2))
mape = np.mean(np.abs((reg_pred_rupiah - y_r_test) / (y_r_test + 1))) * 100

print(f'MAE  : Rp {mae:>12,.0f}')
print(f'RMSE : Rp {rmse:>12,.0f}')
print(f'MAPE : {mape:.2f}%')

## 8. Simpan Model (Siap Produksi)
Menyimpan model dalam format `.keras` — format resmi TensorFlow yang mendukung custom layer.

In [ ]:
# ── SIMPAN MODEL FINAL ────────────────────────────────────────────

# Format .keras (direkomendasikan untuk TF 2.x)
best_model.save('smartspend_model_final.keras')
print('✅ Model disimpan: smartspend_model_final.keras')

# Format SavedModel (untuk deployment ke TF Serving / backend)
best_model.export('smartspend_savedmodel')
print('✅ Model disimpan: smartspend_savedmodel/ (SavedModel format)')

# Simpan konfigurasi metadata
model_metadata = {
    'model_name'    : 'SmartSpend_DL_Model',
    'version'       : '1.0.0',
    'input_features': FEATURE_COLS,
    'input_dim'     : X_train.shape[1],
    'output': {
        'classification': {
            'name'   : 'classification_output',
            'classes': list(le_label.classes_)
        },
        'regression': {
            'name'     : 'regression_output',
            'unit'     : 'Rupiah (setelah invers log1p)',
            'log_scale': True
        }
    }
}

with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2, ensure_ascii=False)

print('✅ Metadata disimpan: model_metadata.json')
print()
print('📁 File yang dihasilkan:')
for f in ['smartspend_best_model.keras', 'smartspend_model_final.keras',
          'smartspend_savedmodel', 'scaler.pkl', 'label_encoders.pkl',
          'label_encoder_target.pkl', 'provinsi_freq.pkl',
          'model_metadata.json', 'training_history.json']:
    exists = '✅' if os.path.exists(f) else '❌'
    print(f'  {exists} {f}')

## 9. Kode Inference (Siap Produksi)
Fungsi inference ini akan dipanggil oleh backend Express.js untuk memberikan prediksi kepada pengguna SmartSpend.

In [ ]:
# ============================================================
# INFERENCE PIPELINE — Siap Integrasi ke Backend
# ============================================================

class SmartSpendInference:
    """
    Kelas inference SmartSpend.
    Digunakan oleh backend Express.js via endpoint /api/ai/predict

    Usage:
    ------
    engine = SmartSpendInference()
    result = engine.predict(user_data)
    """

    def __init__(self,
                 model_path     ='smartspend_model_final.keras',
                 scaler_path    ='scaler.pkl',
                 le_path        ='label_encoders.pkl',
                 le_target_path ='label_encoder_target.pkl',
                 provinsi_path  ='provinsi_freq.pkl',
                 metadata_path  ='model_metadata.json'):

        print('⏳ Memuat model SmartSpend...')
        self.model = keras.models.load_model(
            model_path,
            custom_objects={'FinancialAttentionLayer': FinancialAttentionLayer}
        )
        self.scaler        = joblib.load(scaler_path)
        self.le_dict       = joblib.load(le_path)
        self.le_label      = joblib.load(le_target_path)
        self.provinsi_freq = joblib.load(provinsi_path)
        with open(metadata_path) as f:
            self.metadata = json.load(f)
        self.feature_cols  = self.metadata['input_features']
        print('✅ Model siap digunakan!')

    def _preprocess(self, user_data: dict) -> np.ndarray:
        """Preprocessing data user sebelum diinference."""
        df_user = pd.DataFrame([user_data])

        # Frequency encoding untuk provinsi
        df_user['provinsi_freq'] = df_user.get('provinsi', pd.Series(['DKI Jakarta'])).map(
            lambda x: self.provinsi_freq.get(x, 0.01)
        )

        # Label encode kolom kategorikal
        cat_cols = ['klasifikasi_wilayah', 'jenis_kelamin', 'pendidikan_terakhir',
                    'status_pekerjaan', 'status_pernikahan']
        for col in cat_cols:
            if col in df_user.columns:
                le = self.le_dict[col]
                val = df_user[col].iloc[0]
                # Handle unseen label
                if val in le.classes_:
                    df_user[col] = le.transform([val])
                else:
                    df_user[col] = 0  # Default ke kelas pertama
            else:
                df_user[col] = 0

        # Pastikan semua fitur tersedia
        for col in self.feature_cols:
            if col not in df_user.columns:
                df_user[col] = 0

        X = df_user[self.feature_cols].values.astype(np.float32)
        X = np.nan_to_num(X, nan=0.0)
        X_scaled = self.scaler.transform(X)
        return X_scaled

    def predict(self, user_data: dict) -> dict:
        """
        Inferensi utama: terima data user, kembalikan prediksi.

        Parameters
        ----------
        user_data : dict
            Data keuangan + demografi pengguna.

        Returns
        -------
        dict dengan kunci:
          - kondisi_keuangan  : str  ('Keuangan Sehat' / 'Cukup Baik' / 'Perlu Perbaikan')
          - confidence        : float (probabilitas prediksi 0-1)
          - probabilities     : dict  (probabilitas tiap kelas)
          - rekomendasi_tabungan : int (nominal tabungan ideal dalam Rupiah)
          - pesan             : str  (pesan motivasi)
        """
        X_input = self._preprocess(user_data)

        # Jalankan prediksi
        clf_probs, reg_log = self.model.predict(X_input, verbose=0)

        # Proses output klasifikasi
        class_idx  = int(np.argmax(clf_probs[0]))
        class_name = self.le_label.inverse_transform([class_idx])[0]
        confidence = float(clf_probs[0][class_idx])
        all_probs  = {
            label: round(float(prob), 4)
            for label, prob in zip(self.le_label.classes_, clf_probs[0])
        }

        # Proses output regresi
        tabungan_ideal = int(np.expm1(reg_log[0][0]))
        tabungan_ideal = max(tabungan_ideal, 0)  # Tidak boleh negatif

        # Generate pesan motivasi
        pesan_map = {
            'Keuangan Sehat'    : '🎉 Keuangan Anda sangat baik! Pertahankan dan tingkatkan investasi.',
            'Cukup Baik'        : '💪 Keuangan Anda cukup baik. Tingkatkan tabungan untuk keamanan lebih.',
            'Perlu Perbaikan'   : '⚠️ Keuangan perlu perhatian. Kurangi pengeluaran tidak perlu.'
        }

        return {
            'kondisi_keuangan'    : class_name,
            'confidence'          : round(confidence, 4),
            'probabilities'       : all_probs,
            'rekomendasi_tabungan': tabungan_ideal,
            'pesan'               : pesan_map.get(class_name, '')
        }


# Inisialisasi inference engine
engine = SmartSpendInference()

In [ ]:
# ── DEMO INFERENCE ────────────────────────────────────────────────
test_users = [
    {
        'nama': 'Budi — Karyawan Swasta (kondisi kritis)',
        'data': {
            'provinsi': 'DKI Jakarta', 'klasifikasi_wilayah': 'Perkotaan',
            'jenis_kelamin': 'Laki-laki', 'usia': 28.0,
            'pendidikan_terakhir': 'S1', 'status_pekerjaan': 'Karyawan Swasta',
            'status_pernikahan': 'Belum Menikah', 'jumlah_tanggungan': 0.0,
            'pendapatan_bulanan': 4500000.0, 'total_pengeluaran': 4350000.0,
            'tabungan_bulanan': 150000.0, 'rasio_tabungan_persen': 3.3,
            'skor_literasi_keuangan': 35.0, 'cicilan_hutang': 800000.0,
            'pengeluaran_makanan_pokok': 400000.0, 'pengeluaran_lauk_pauk': 250000.0,
            'pengeluaran_sayur_buah': 100000.0, 'pengeluaran_jajan_makan_luar': 900000.0,
            'pengeluaran_rokok_tembakau': 300000.0, 'pengeluaran_perumahan_listrik': 750000.0,
            'pengeluaran_pakaian': 200000.0, 'pengeluaran_kesehatan': 100000.0,
            'pengeluaran_pendidikan': 0.0, 'pengeluaran_transportasi': 350000.0,
            'pengeluaran_komunikasi': 100000.0, 'pengeluaran_hiburan_rekreasi': 400000.0
        }
    },
    {
        'nama': 'Sari — PNS (kondisi sehat)',
        'data': {
            'provinsi': 'Jawa Barat', 'klasifikasi_wilayah': 'Perkotaan',
            'jenis_kelamin': 'Perempuan', 'usia': 35.0,
            'pendidikan_terakhir': 'S1', 'status_pekerjaan': 'PNS/ASN',
            'status_pernikahan': 'Menikah', 'jumlah_tanggungan': 2.0,
            'pendapatan_bulanan': 9000000.0, 'total_pengeluaran': 6000000.0,
            'tabungan_bulanan': 3000000.0, 'rasio_tabungan_persen': 33.3,
            'skor_literasi_keuangan': 75.0, 'cicilan_hutang': 1500000.0,
            'pengeluaran_makanan_pokok': 600000.0, 'pengeluaran_lauk_pauk': 400000.0,
            'pengeluaran_sayur_buah': 200000.0, 'pengeluaran_jajan_makan_luar': 500000.0,
            'pengeluaran_rokok_tembakau': 0.0, 'pengeluaran_perumahan_listrik': 1200000.0,
            'pengeluaran_pakaian': 300000.0, 'pengeluaran_kesehatan': 300000.0,
            'pengeluaran_pendidikan': 500000.0, 'pengeluaran_transportasi': 400000.0,
            'pengeluaran_komunikasi': 150000.0, 'pengeluaran_hiburan_rekreasi': 150000.0
        }
    }
]

for user in test_users:
    print('\n' + '─' * 55)
    print(f"👤 {user['nama']}")
    result = engine.predict(user['data'])
    print(f"   Kondisi Keuangan    : {result['kondisi_keuangan']}")
    print(f"   Confidence          : {result['confidence']*100:.1f}%")
    print(f"   Probabilities       : {result['probabilities']}")
    print(f"   Rekomendasi Tabungan: Rp {result['rekomendasi_tabungan']:,}")
    print(f"   Pesan               : {result['pesan']}")

## 10. Ringkasan Checklist MVP

| No | Requirement | Status | Implementasi |
|---|---|---|---|
| 1 | TensorFlow Functional API / Model Subclassing | ✅ | `keras.Input` + `Model(inputs, outputs)` |
| 2 | Custom Layer | ✅ | `FinancialAttentionLayer` (attention mechanism) |
| 3 | Custom Callback | ✅ | `SmartSpendCallback` (monitor + auto-save + early stop) |
| 4 | Simpan model `.keras` | ✅ | `smartspend_model_final.keras` + `smartspend_savedmodel/` |
| 5 | Kode inference | ✅ | `SmartSpendInference` class siap integrasi ke Express.js |

---

**Cara integrasi ke backend Express.js:**
```javascript
// routes/ai.js
const { PythonShell } = require('python-shell');

router.post('/predict', async (req, res) => {
    const userData = req.body;
    const options = {
        mode: 'json',
        scriptPath: './ai',
        args: [JSON.stringify(userData)]
    };
    PythonShell.run('inference_api.py', options, (err, results) => {
        if (err) return res.status(500).json({ error: err.message });
        res.json(results[0]);
    });
});
```